# Read Streaming Data

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
schema=StructType([
    StructField('order_id', StringType()),
    StructField('timestamp', TimestampType()),
    StructField('customer',StructType([
        StructField('customer_id',IntegerType()),
        StructField('name', StringType()),
        StructField('email', StringType()),
        StructField('address', StructType([
            StructField('city', StringType()),
            StructField('country', StringType()),
            StructField('postal_code', StringType())
        ]))
    ])),
    StructField('items',ArrayType(
        StructType([
            StructField('item_id' , StringType()),
            StructField('price', DoubleType()),
            StructField('product_name',StringType()),
            StructField('quantity',IntegerType())
        ])
    )),
    StructField('metadata', ArrayType(
        StructType([
            StructField('key', StringType()),
            StructField('value', StringType())
        ])
    )),
    StructField('payment', StructType([
        StructField('method', StringType()),
        StructField('transaction_id', StringType())
    ]))
])

In [0]:
df=spark.readStream \
        .format('json') \
        .option("multiLine",True) \
        .schema(schema) \
        .load("/Volumes/mycata/stream/streaming/source")

In [0]:
df.printSchema()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5792964234785723>, line 1
----> 1 df.printSchema()

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:2015, in DataFrame.printSchema(self, level)
   2013     print(self.schema.treeString(level))
   2014 else:
-> 2015     print(self.schema.treeString())

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1988, in DataFrame.schema(self)
   1985 @property
   1986 def schema(self) -> StructType:
   1987     # self._schema call will cache the schema and serialize it if it is not cached yet.
-> 1988     _schema = self._schema
   1989     if self._cached_schema_serialized is not None:
   1990         try:

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/dataframe.py:1977, in DataFrame._schema(self)
   1975 if self._cached_schema is Non

### Flattening json

In [0]:
df=df.select(
          'order_id',
          'customer.customer_id',
          'customer.name',
          'customer.email',
          'customer.address.city',
          'customer.address.country',
          'customer.address.postal_code',
           explode_outer('items').alias('items'),
           explode_outer('metadata').alias('metadata'),
          'payment.method',
          'payment.transaction_id',
          'timestamp'
          )

In [0]:
df=df.select('order_id',
          'customer_id',
          'name',
          'email',
          'city',
          'country',
          'postal_code',
          'items.item_id',
          'items.price',
          'items.product_name',
          'items.quantity',
          'metadata.key',
          'metadata.value',
          'method',
          'transaction_id',
          'timestamp'
          )

NOTE: outputMode: Used only in Structured Streaming (writeStream) to define how data is written to the sink when new data arrives (e.g., writing only new rows, or rewriting the whole table).

In [0]:
df.writeStream \
    .format('delta') \
    .outputMode('append') \
    .option('checkpointLocation', '/Volumes/mycata/stream/streaming/sink/sink_Checpoint' ) \
    .option('path', '/Volumes/mycata/stream/streaming/sink/data') \
    .trigger(once=True) \
    .start()

In [0]:
%sql
SELECT * FROM delta.`/Volumes/mycata/stream/streaming/sink/data`;